# Milestone 1 arXiv Crawler trên Google Colab — v3.13.0

Cào toàn bộ dải ID (`id_ranges.csv`) trên **1 máy duy nhất** (tuân thủ nghiêm ngặt Quyết định D8 & rubric đồ án).
Lưu `S2_API_KEY` trong mục **Secrets** của Colab (khóa `S2_API_KEY`, không bao giờ hardcode vào notebook).

---

### Kiến trúc 2 giai đoạn (Two-Stage Decoupled Protocol — v3.13)
Nhằm giải quyết triệt để tình trạng nghẽn cổ chai và bão cooldown khi cào đồng thời nhiều dịch vụ:

1. **Giai đoạn 1 (Tải TeX Source arXiv)**: Dùng cờ `--skip-s2` để dồn 100% tài nguyên và socket cho `export.arxiv.org`. Tốc độ cào đạt mức sàn lý thuyết (~39.6h cho ~47.500 versions), không bị Semantic Scholar ngắt nhịp hay kéo dài thời gian.
   - *Tùy chọn tăng tốc cực đại*: Nếu có tệp Kaggle Snapshot (`arxiv-metadata-oai-snapshot.json`), thêm cờ `--snapshot` để lọc metadata offline trong < 60s, triệt tiêu 100% request Atom API qua mạng.
2. **Giai đoạn 2 (Bổ sung trích dẫn Semantic Scholar)**: Dùng cờ `--s2-only --enable-bib-fallback` sau khi Giai đoạn 1 hoàn tất. Hệ thống chỉ quét các bài còn thiếu trích dẫn và gọi S2 Graph API (theo lô 25 bài), tự động fallback sang tệp `.bib/.bbl` có sẵn nếu S2 404.

---

### Vòng lặp phiên chạy an toàn: Chạy khi rảnh, dừng khi bận — không mất dữ liệu!
Dữ liệu được lưu trữ an toàn trên đĩa cục bộ Colab (`/content/data`) và tự động đồng bộ lên Google Drive (`arxiv-data.tar.gz`):

1. **Setup**: Clone / pull mã nguồn, mount Google Drive, cấu hình tham số.
2. **Restore**: Bung backup `arxiv-data.tar.gz` từ Google Drive vào `/content/data` (chỉ mất vài giây).
3. **Crawl**: Chạy lệnh cào. Hệ thống tự động đọc `manifest.sqlite`, **bỏ qua 100% các bài/version đã hoàn tất**, chỉ cào phần còn thiếu. Tự động lưu backup nền mỗi 1000 ID và khi dừng.
4. **Dừng an toàn**: Bấm **Runtime → Interrupt execution**, sau đó chạy cell **Backup** (`backup_now(unmount=True)`) trước khi đóng tab.
5. **Phiên tiếp theo**: Lặp lại từ bước 1.


In [ ]:
import os

if not os.path.isdir("/content/arxiv-pipeline"):
    !git clone https://github.com/GiaaPhuc/arxiv-pipeline.git
else:
    !git -C /content/arxiv-pipeline pull --ff-only
%cd /content/arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import logging
import os
import sys
import time
from pathlib import Path

from google.colab import drive, userdata

if "/content/arxiv-pipeline/src" not in sys.path:
    sys.path.insert(0, "/content/arxiv-pipeline/src")

from crawler.checkpoint import (
    backup_tar_archive,
    is_workspace_complete,
    manifest_pair_counts,
    read_archive_meta,
    restore_data_dir,
)

_DRIVE_MOUNT = "/content/drive"
_DATA_DIR = Path("/content/data")
_BACKUP_TAR_GZ = Path("/content/drive/MyDrive/arxiv-data.tar.gz")
_BACKUP_TAR = Path("/content/drive/MyDrive/arxiv-data.tar")
_BACKUP_LEGACY_DIR = Path("/content/drive/MyDrive/arxiv-data")
_SNAPSHOT_PATH = Path("/content/drive/MyDrive/arxiv-metadata-oai-snapshot.json")


def _drive_usable() -> bool:
    my_drive = os.path.join(_DRIVE_MOUNT, "MyDrive")
    if not os.path.isdir(my_drive):
        return False
    try:
        os.listdir(my_drive)
        return True
    except OSError:
        return False


if _drive_usable():
    print("Google Drive already mounted.")
else:
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.isdir(_DRIVE_MOUNT) and os.listdir(_DRIVE_MOUNT):
        raise RuntimeError(
            "Cannot mount Drive: /content/drive still has files "
            "(common after disconnect). Runtime → Restart session, "
            "then run this notebook from the top."
        )
    drive.mount(_DRIVE_MOUNT)

# Configuration & API Secrets
os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
os.environ["CRAWLER_CONTACT"] = "your-email@example.com"  # User-Agent contact
os.environ["CRAWLER_DATA_DIR"] = str(_DATA_DIR)
os.environ["CRAWLER_BACKUP_DIR"] = str(_BACKUP_TAR_GZ)
os.environ["CRAWLER_BACKUP_EVERY_IDS"] = "1000"  # Background backup every 1000 IDs
os.environ["CRAWLER_ARXIV_MIN_INTERVAL"] = "3.35"  # 3.35s floor (an toan gio cao diem)
os.environ["CRAWLER_ARXIV_MIN_IDLE_GAP"] = "3.25"  # 3.25s End-to-Start gap
os.environ["CRAWLER_ARXIV_PROACTIVE_COOLDOWN_EVERY_IDS"] = "120"  # 30s breather / 120 IDs
os.environ["CRAWLER_ARXIV_PROACTIVE_COOLDOWN_SECONDS"] = "30"
os.environ["CRAWLER_AUTO_RESUME_RATE_LIMIT"] = "1"  # Auto-defer & cooldown on 429
os.environ["CRAWLER_ENABLE_BIB_FALLBACK"] = "1"  # Dual-engine local .bib fallback
os.environ["CRAWLER_ENABLE_OAI_DATES"] = "1"  # OAI-PMH arXivRaw date recovery
os.environ["CRAWLER_S2_BATCH_SIZE"] = "25"  # Batch 25 prevents 413/504 payload error

if _SNAPSHOT_PATH.is_file():
    print(f"Kaggle Snapshot found on Drive: {_SNAPSHOT_PATH.name} (ready for offline discovery)")
else:
    print("Kaggle Snapshot not found on Drive (will use online arXiv Atom discovery by default).")


def _archive_rank(path: Path) -> tuple[int, float]:
    meta = read_archive_meta(path)
    raw_pairs = meta.get("final_pairs") if meta else None
    pairs = int(raw_pairs) if isinstance(raw_pairs, (int, float)) else -1
    return (pairs, path.stat().st_mtime)


def backup_now(*, allow_shrink: bool = False, unmount: bool = False) -> None:
    """Pack /content/data into one atomic archive on Drive (with shrink guard)."""
    if not _drive_usable():
        raise RuntimeError("Google Drive is not mounted; re-run the Setup cell first.")
    if not (_DATA_DIR / "manifest.sqlite").is_file():
        print("Nothing to back up yet (/content/data/manifest.sqlite not found).")
        return
    info = backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ, allow_shrink=allow_shrink)
    mib = info["archive_bytes"] / (1024 * 1024)
    print(
        f"backup done {time.strftime('%H:%M:%S')}: {_BACKUP_TAR_GZ.name} "
        f"({mib:.1f} MiB, {info['final_pairs']} final pairs)"
    )
    if unmount:
        drive.flush_and_unmount()
        print("Drive flushed to cloud and unmounted — safe to close Colab.")


def restore_now(*, force: bool = False) -> None:
    """Restore /content/data from Drive archive (or migrate legacy folder once)."""
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    if not _drive_usable():
        raise RuntimeError("Google Drive is not mounted; re-run the Setup cell first.")
    if not force and is_workspace_complete(_DATA_DIR):
        local_final, _ = manifest_pair_counts(_DATA_DIR / "manifest.sqlite")
        print(
            f"Skip restore: {_DATA_DIR} already has complete local data "
            f"({local_final} final pairs). Pass restore_now(force=True) to overwrite."
        )
        return

    archives = [p for p in (_BACKUP_TAR_GZ, _BACKUP_TAR) if p.is_file()]
    if archives:
        chosen = max(archives, key=_archive_rank)
        info = restore_data_dir(chosen, _DATA_DIR, force=force)
        print(f"Restored {chosen.name} -> {_DATA_DIR} ({info['final_pairs']} final pairs).")
        if chosen != _BACKUP_TAR_GZ:
            backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ)
            print(f"Migrated {chosen.name} -> {_BACKUP_TAR_GZ.name}.")
        return

    has_legacy = _BACKUP_LEGACY_DIR.is_dir() and (
        (_BACKUP_LEGACY_DIR / "manifest.sqlite").is_file()
        or (_BACKUP_LEGACY_DIR / "papers").is_dir()
    )
    if has_legacy:
        print(
            "Legacy folder backup found on Drive: running 16-thread parallel restore "
            "and migrating to arxiv-data.tar.gz..."
        )
        info = restore_data_dir(
            _BACKUP_LEGACY_DIR,
            _DATA_DIR,
            force=force,
            legacy_workers=16,
            progress_every=250,
        )
        if (_DATA_DIR / "manifest.sqlite").is_file():
            bk = backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ)
            mib = bk["archive_bytes"] / (1024 * 1024)
            print(
                f"Migrated legacy folder -> {_BACKUP_TAR_GZ.name} "
                f"({mib:.1f} MiB, {info['final_pairs']} final pairs). "
                "Next restore will take seconds!"
            )
        return

    _DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("No backup on Drive (first run).")

In [ ]:
# [TÙY CHỌN NÂNG CAO] Tải Kaggle Snapshot về Google Drive (chỉ cần chạy 1 lần duy nhất)
# Yêu cầu: Đã upload tệp kaggle.json vào MyDrive hoặc thư mục gốc Drive
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d Cornell-University/arxiv -p /content/drive/MyDrive/ --unzip

In [ ]:
# Plan only: kiểm tra danh sách ID và cấu hình (không gọi mạng, không ghi file)
!python run_crawl.py --ranges id_ranges.csv --dry-run

In [ ]:
# Restore from Drive — Chạy MỖI KHI bắt đầu một session Colab mới.
# - Tự động giải nén arxiv-data.tar.gz từ Drive vào /content/data trong vài giây.
# - Dữ liệu cũ đã crawl sẽ được phục hồi nguyên vẹn và KHÔNG BAO GIỜ bị cào lại!
restore_now()

### Hướng dẫn chọn lệnh Crawl

Tùy theo giai đoạn công việc, chọn **1 trong các lệnh** dưới cell tiếp theo:

1. **GIAI ĐOẠN 1 (Khuyến nghị v3.13 — Bắt đầu cào TeX source)**:
   - Chạy với cờ `--skip-s2` để đạt tốc độ tối đa từ `export.arxiv.org`.
   - Nếu có file Kaggle Snapshot trên Drive, thêm `--snapshot /content/drive/MyDrive/arxiv-metadata-oai-snapshot.json`.
   - Tự động bỏ qua mọi bài đã hoàn thành từ các phiên trước.

2. **GIAI ĐOẠN 2 (Cào bù trích dẫn Semantic Scholar sau khi Giai đoạn 1 đã xong)**:
   - Chạy với cờ `--s2-only --enable-bib-fallback`.
   - Không gọi arXiv, chỉ gọi S2 theo lô 25 bài và tự động trích xuất `.bib/.bbl` có sẵn.

3. **Chạy thử nghiệm (Pilot 30 bài)**:
   - Dành cho lần đầu tiên kiểm tra toàn bộ pipeline từ đầu đến cuối.


In [ ]:
# ==============================================================================
# LỆNH 1: [GIAI ĐOẠN 1 - KHUYẾN NGHỊ v3.13] CÀO NHANH NGUỒN ARXIV (BỎ QUA S2)
# ==============================================================================
# Tốc độ tối đa trên 1 máy, loại bỏ hoàn toàn bão cooldown từ Semantic Scholar.
# Manifest tự động bỏ qua mọi bài/version đã cào từ phiên trước.
!python run_crawl.py --ranges id_ranges.csv \
    --skip-s2 \
    --auto-resume-rate-limit \
    --enable-oai-dates

# ------------------------------------------------------------------------------
# LỆNH 1B: [GIAI ĐOẠN 1 + KAGGLE SNAPSHOT] NẾU CÓ TỆP SNAPSHOT TRÊN GOOGLE DRIVE
# ------------------------------------------------------------------------------
# Tiết kiệm 100% thời gian gọi Atom API (< 60s nạp metadata offline cho 25.000 ID)
# !python run_crawl.py --ranges id_ranges.csv \
#     --snapshot /content/drive/MyDrive/arxiv-metadata-oai-snapshot.json \
#     --skip-s2 \
#     --auto-resume-rate-limit \
#     --enable-oai-dates

# ==============================================================================
# LỆNH 2: [GIAI ĐOẠN 2] CÀO BÙ SEMANTIC SCHOLAR (CHỈ CHẠY KHI GIAI ĐOẠN 1 ĐÃ XONG)
# ==============================================================================
# Chỉ cào bổ sung trích dẫn S2, kèm dual-engine BibTeX fallback (không chạm vào arXiv)
# !python run_crawl.py --ranges id_ranges.csv \
#     --s2-only \
#     --enable-bib-fallback

# ==============================================================================
# LỆNH 3: [PILOT] CHẠY THỬ 30 BÀI ĐẦU TIÊN ĐỂ KIỂM TRA MÔI TRƯỜNG
# ==============================================================================
# !python run_crawl.py --ranges id_ranges.csv --limit 30 \
#     --auto-resume-rate-limit --enable-bib-fallback --enable-oai-dates

### Quy trình dừng / chạy tiếp (Pause & Resume)

**A. Dừng tạm thời nhưng runtime Colab vẫn còn sống** (khi muốn nghỉ hoặc hết giờ):
1. Bấm nút **Runtime → Interrupt execution** (dừng mềm). Crawler sẽ tự động hoàn tất bài đang tải dở và lưu bản backup cuối cùng vào `arxiv-data.tar.gz`.
2. Để chắc chắn dữ liệu được đồng bộ an toàn lên Google Drive trước khi đóng tab: chạy cell **Backup** bên dưới với `backup_now(unmount=True)`.
3. Nếu muốn chạy tiếp ngay trong cùng session: chỉ cần bấm chạy lại cell **Crawl** (không cần chạy lại Restore).

**B. Bị ngắt kết nối / Mất runtime (Disconnect)**:
1. Mở lại notebook, chạy lần lượt từ trên xuống:
   - Cell 1: Clone / Pull code.
   - Cell 2: Setup & Mount Drive.
   - Cell 5: Restore (`restore_now()`) — toàn bộ dữ liệu cào trước đó sẽ được giải nén trở lại `/content/data` trong vài giây.
   - Cell 7: Crawl — tiếp tục cào từ bài đang dang dở.

**C. Ý nghĩa các mã thoát (Exit codes)**:
- `Exit code 0`: Đã hoàn thành 100% toàn bộ dải bài được giao.
- `Exit code 3`: arXiv đang tạm thời bận hoặc siết rate limit sau nhiều lần thử lại. Hệ thống đã tự động lưu dữ liệu an toàn. Hãy tạm dừng nghỉ 30–60 phút rồi chạy lại lệnh để tiếp tục.


In [ ]:
# Backup dữ liệu thủ công lên Drive (khuyên dùng unmount=True trước khi tắt máy)
backup_now()
# backup_now(unmount=True)

In [ ]:
# ==============================================================================
# KIỂM TRA & XUẤT BẢNG DỮ LIỆU (Chỉ chạy khi Giai đoạn 1 và 2 đã hoàn tất 100%)
# ==============================================================================

# 1. Kiểm tra tính toàn vẹn kho dữ liệu theo rubric đồ án (không thiếu file, không corrupt):
# !python validate_corpus.py --data-dir $CRAWLER_DATA_DIR

# 2. Kiểm toán đối soát kho tệp trên đĩa so với bảng manifest.sqlite:
# !python audit_corpus.py --data-dir $CRAWLER_DATA_DIR --output audit_report.json

# 3. Làm giàu trích dẫn offline từ các tệp .bib/.bbl có sẵn (không tốn mạng):
# !python enrich_references.py --data-dir $CRAWLER_DATA_DIR

# 4. Xuất dữ liệu ra bảng Parquet (papers.parquet, references.parquet) và thống kê:
# !python export_data.py --data-dir $CRAWLER_DATA_DIR